# Deliberate Practice Notebook: Async Python & FastAPI (15-Hour Intensive)

**Target Practice Time:** 15 Hours (900 Minutes)  
**Exercise Volume:** 16 Exercises (Calculated at 1 exercise per ~56 minutes of deep implementation, debugging, and profiling)  
**Execution Environment:** Google Colab / Jupyter (Top-level `await` and `nest_asyncio` enabled).

### Syllabus Coverage
1. **Level 1 (Core Mechanics & Syntax):** Event loops, non-blocking thread offloading (`asyncio.to_thread`), `asyncio.TaskGroup` structured concurrency, Pydantic v2 `strict=True` validators, and Generic models (`Generic[T]`, `@computed_field`).
2. **Level 2 (Real-World Implementation):** `asyncio.Semaphore` with exponential backoff, bounded `asyncio.Queue` backpressure pipelines, FastAPI `lifespan` state management, Server-Sent Events (SSE) with disconnect cleanup, WebSockets with dead-socket pruning, and zero-overhead NDJSON streaming with `TypeAdapter`.
3. **Level 3 (Edge Cases, Performance & Failure Modes):** Event loop lag watchdog timers, automated CPU bottleneck detection (`cProfile` / `pstats`), memory leak diagnosis (`tracemalloc` & weak/callback task cleanup), atomic transaction protection (`asyncio.shield`), and fast-fail queue load shedding.
4. **Level 4 (Milestone Mini-Project):** Full Multi-Provider Streaming LLM Gateway Microservice with automatic failover, strict validation, concurrency shedding, and live telemetry.

In [3]:
# [COLAB SETUP CELL]
!pip install -q fastapi uvicorn httpx "pydantic>=2.8.0" websockets sse-starlette nest_asyncio py-spy

import asyncio
import time
import tracemalloc
import cProfile
import pstats
import io
import json
import gc
import hashlib
from collections import deque
from contextlib import asynccontextmanager
from typing import Generic, TypeVar, List, Dict, Any, Optional, AsyncGenerator, Callable, Tuple, Union, Literal, Annotated
import nest_asyncio

# Apply nest_asyncio to allow nested event loops in Google Colab
nest_asyncio.apply()
print("Environment initialized successfully. Pydantic v2, FastAPI, and Asyncio ready.")

Environment initialized successfully. Pydantic v2, FastAPI, and Asyncio ready.


---
# Level 1: Core Mechanics & Syntax (Warm-Up)

## Exercise 1: Event Loop Blocking vs. Non-Blocking Offloading
**Estimated Time:** 20 mins  
**Context:** A single synchronous CPU-bound or blocking I/O call (like `time.sleep` or synchronous tokenization) inside an `async def` function freezes the entire Python event loop, destroying concurrency for all connected users.  
**Problem Statement:** Implement `run_concurrent_tasks(n_io: int, n_blocking: int) -> List[str]`. You must run `n_io` calls to `async_io_task()` and `n_blocking` calls to `legacy_blocking_task()` concurrently. Use `asyncio.to_thread` to offload the legacy synchronous function so the total execution time remains under `0.35` seconds for 5 of each task (each task takes `0.2s`).  
**Constraints:** Do not modify `legacy_blocking_task` or `async_io_task`. Return the combined list of outputs.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 1]
import asyncio
import time
from typing import List

def legacy_blocking_task(task_id: int) -> str:
    """Simulates a legacy C-extension or blocking file/network operation."""
    time.sleep(0.2)
    return f"blocking_{task_id}"

async def async_io_task(task_id: int) -> str:
    """Simulates a non-blocking network request."""
    await asyncio.sleep(0.2)
    return f"async_{task_id}"

print(async_io_task(1))

<coroutine object async_io_task at 0x0000020EC1C62330>


D:\temp\ipykernel_28328\2629331039.py:16: RuntimeWarning: coroutine 'async_io_task' was never awaited
  print(async_io_task(1))


In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 1]
async def run_concurrent_tasks(n_io: int, n_blocking: int) -> List[str]:
    """
    Executes n_io async tasks and n_blocking sync tasks concurrently without
    blocking the main event loop. Returns the combined list of result strings.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 1]
async def validate_ex1():
    start = time.perf_counter()
    results = await run_concurrent_tasks(5, 5)
    elapsed = time.perf_counter() - start
    
    assert isinstance(results, list), "Output must be a list."
    assert len(results) == 10, f"Expected 10 results, got {len(results)}"
    assert "blocking_0" in results and "async_4" in results, "Missing expected task outputs."
    assert elapsed < 0.35, f"Event loop was blocked! Elapsed time: {elapsed:.3f}s (expected < 0.35s)"
    print(f"PASS: Exercise 1 completed in {elapsed:.3f}s.")

await validate_ex1()

<details><summary>Click to view solution (Exercise 1)</summary>

```python
async def run_concurrent_tasks(n_io: int, n_blocking: int) -> List[str]:
    io_coros = [async_io_task(i) for i in range(n_io)]
    blocking_coros = [asyncio.to_thread(legacy_blocking_task, i) for i in range(n_blocking)]
    return list(await asyncio.gather(*io_coros, *blocking_coros))
```
</details>

---
## Exercise 2: Structured Concurrency with `asyncio.TaskGroup`
**Estimated Time:** 25 mins  
**Context:** `asyncio.gather` leaves orphan background tasks running if one task raises an exception unless you manually cancel and await them. Python 3.11+ introduced `asyncio.TaskGroup` for strict structured concurrency: if one subtask fails, all sibling tasks in the group are automatically cancelled and awaited before propagating the `ExceptionGroup`.  
**Problem Statement:** Implement `fetch_all_or_cancel(urls: List[str], tracker: TaskTracker) -> List[str]`. Use `asyncio.TaskGroup` to schedule `simulated_fetch(url, tracker)` for each URL. If any URL contains the substring `"fail"`, `simulated_fetch` raises a `ValueError`. Catch the `ExceptionGroup` using `except* ValueError`, return `["ABORTED"]`, and verify that all sibling tasks were cleanly cancelled without leaking background execution.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 2]
import asyncio
from typing import List

class TaskTracker:
    def __init__(self):
        self.active = 0
        self.cancelled = 0
        self.completed = 0

async def simulated_fetch(url: str, tracker: TaskTracker) -> str:
    tracker.active += 1
    try:
        if "fail" in url:
            await asyncio.sleep(0.05)
            raise ValueError(f"Connection reset for {url}")
        await asyncio.sleep(0.3)
        tracker.completed += 1
        return f"200 OK: {url}"
    except asyncio.CancelledError:
        tracker.cancelled += 1
        raise
    finally:
        tracker.active -= 1

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 2]
async def fetch_all_or_cancel(urls: List[str], tracker: TaskTracker) -> List[str]:
    """
    Executes simulated_fetch for all URLs inside an asyncio.TaskGroup.
    If ValueError occurs inside the group, catch it via except* ValueError and return ["ABORTED"].
    Otherwise, return the list of results in order.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 2]
async def validate_ex2():
    tracker1 = TaskTracker()
    res1 = await fetch_all_or_cancel(["http://a.com", "http://b.com"], tracker1)
    assert res1 == ["200 OK: http://a.com", "200 OK: http://b.com"], f"Unexpected output: {res1}"
    assert tracker1.completed == 2 and tracker1.cancelled == 0

    tracker2 = TaskTracker()
    res2 = await fetch_all_or_cancel(["http://a.com", "http://fail.com", "http://c.com"], tracker2)
    assert res2 == ["ABORTED"], f"Expected ['ABORTED'], got {res2}"
    assert tracker2.active == 0, "Orphan tasks are still running!"
    assert tracker2.cancelled == 2, f"Expected 2 cancelled sibling tasks, got {tracker2.cancelled}"
    assert tracker2.completed == 0, "Slow tasks should not have completed after sibling failure."
    print("PASS: Exercise 2 passed structured cancellation checks.")

await validate_ex2()

<details><summary>Click to view solution (Exercise 2)</summary>

```python
async def fetch_all_or_cancel(urls: List[str], tracker: TaskTracker) -> List[str]:
    tasks = []
    try:
        async with asyncio.TaskGroup() as tg:
            for url in urls:
                tasks.append(tg.create_task(simulated_fetch(url, tracker)))
        return [t.result() for t in tasks]
    except* ValueError:
        return ["ABORTED"]
```
</details>

---
## Exercise 3: Pydantic v2 Strict Mode & Custom Field/Model Validators
**Estimated Time:** 25 mins  
**Context:** By default, Pydantic coerces types (e.g., converting the string `"100"` into integer `100`, or `1` into `True`). In high-reliability LLM tool-calling pipelines, silent type coercion hides hallucinations. Pydantic v2 offers `ConfigDict(strict=True)` along with Rust-backed `@field_validator` and `@model_validator`.  
**Problem Statement:** Define a Pydantic v2 model `LLMGenerationConfig` with strict validation enabled (`strict=True`, `extra='forbid'`):
- `model_name`: `str` (must start with `"gpt-"` or `"claude-"`; strip leading/trailing whitespace via a `'before'` field validator, while still rejecting non-string inputs like integers or lists before stripping).
- `temperature`: `float` (strictly `0.0 <= temperature <= 2.0`).
- `max_tokens`: `int` (strictly `> 0`).
- `top_k`: `Optional[int] = None`.
- Add a `@model_validator(mode='after')` enforcing that if `temperature == 0.0` (greedy decoding), `top_k` MUST be `None` or `1`, otherwise raise a `ValueError("Greedy decoding requires top_k to be None or 1")`.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 3]
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator, ValidationError
from typing import Optional, Any

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 3]
class LLMGenerationConfig(BaseModel):
    # YOUR CODE HERE
    pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 3]
def validate_ex3():
    cfg = LLMGenerationConfig(model_name="  claude-3-5-sonnet ", temperature=0.7, max_tokens=512, top_k=40)
    assert cfg.model_name == "claude-3-5-sonnet"
    
    try:
        LLMGenerationConfig(model_name="gpt-4o", temperature=0.5, max_tokens="512")
        assert False, "Should have rejected string for max_tokens in strict mode"
    except ValidationError:
        pass

    try:
        LLMGenerationConfig(model_name=12345, temperature=0.5, max_tokens=100)
        assert False, "Should have rejected int for model_name"
    except ValidationError:
        pass

    try:
        LLMGenerationConfig(model_name="gpt-4o", temperature=0.0, max_tokens=100, top_k=50)
        assert False, "Should have rejected temperature=0.0 with top_k=50"
    except ValidationError as e:
        assert "Greedy decoding" in str(e)

    try:
        LLMGenerationConfig(model_name="gpt-4o", temperature=0.5, max_tokens=100, unknown_arg=True)
        assert False, "Should have forbidden extra fields"
    except ValidationError:
        pass

    print("PASS: Exercise 3 strict Pydantic v2 validation verified.")

validate_ex3()

<details><summary>Click to view solution (Exercise 3)</summary>

```python
class LLMGenerationConfig(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")

    model_name: str
    temperature: float = Field(ge=0.0, le=2.0)
    max_tokens: int = Field(gt=0)
    top_k: Optional[int] = None

    @field_validator("model_name", mode="before")
    @classmethod
    def sanitize_and_validate_prefix(cls, v: Any) -> str:
        if not isinstance(v, str):
            raise ValueError("model_name must be a string")
        cleaned = v.strip()
        if not (cleaned.startswith("gpt-") or cleaned.startswith("claude-")):
            raise ValueError("model_name must start with 'gpt-' or 'claude-'")
        return cleaned

    @model_validator(mode="after")
    def verify_greedy_decoding(self) -> "LLMGenerationConfig":
        if self.temperature == 0.0 and self.top_k not in (None, 1):
            raise ValueError("Greedy decoding requires top_k to be None or 1")
        return self
```
</details>

---
## Exercise 4: Pydantic v2 Generic Models & Computed Properties
**Estimated Time:** 20 mins  
**Context:** Microservices returning standardized API envelopes for diverse LLM payloads (embeddings, chat completions, moderation flags) require reusable Generic Pydantic models with zero-overhead serialization.  
**Problem Statement:** Create a generic Pydantic v2 model `APIEnvelope(BaseModel, Generic[DataT])` containing:
- `status`: `str`
- `payload`: `DataT`
- `latency_ms`: `float`
- `tokens_used`: `int`
- A `@computed_field` property named `tokens_per_second` (`float`) that computes `round(tokens_used / (latency_ms / 1000.0), 2)` (return `0.0` if `latency_ms <= 0`). Ensure `tokens_per_second` is automatically included when `.model_dump()` or `.model_dump_json()` is called.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 4]
from pydantic import BaseModel, computed_field, ValidationError
from typing import Generic, TypeVar, List

DataT = TypeVar("DataT")

class EmbeddingPayload(BaseModel):
    vector: List[float]
    dimensions: int

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 4]
class APIEnvelope(BaseModel, Generic[DataT]):
    # YOUR CODE HERE
    pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 4]
def validate_ex4():
    emb = EmbeddingPayload(vector=[0.1, 0.2, 0.3], dimensions=3)
    env = APIEnvelope[EmbeddingPayload](
        status="success",
        payload=emb,
        latency_ms=500.0,
        tokens_used=125
    )
    dumped = env.model_dump()
    assert dumped["tokens_per_second"] == 250.0, f"Expected 250.0, got {dumped.get('tokens_per_second')}"
    assert dumped["payload"]["dimensions"] == 3

    try:
        APIEnvelope[EmbeddingPayload](
            status="success",
            payload={"wrong_field": "invalid"},
            latency_ms=100.0,
            tokens_used=10
        )
        assert False, "Generic payload validation failed to catch invalid inner type!"
    except ValidationError:
        pass

    print("PASS: Exercise 4 Generic Pydantic v2 model and computed_field verified.")

validate_ex4()

<details><summary>Click to view solution (Exercise 4)</summary>

```python
class APIEnvelope(BaseModel, Generic[DataT]):
    status: str
    payload: DataT
    latency_ms: float
    tokens_used: int

    @computed_field
    @property
    def tokens_per_second(self) -> float:
        if self.latency_ms <= 0:
            return 0.0
        return round(self.tokens_used / (self.latency_ms / 1000.0), 2)
```
</details>

---
# Level 2: Real-World Implementation (Data Pipelines & Microservices)

## Exercise 5: High-Concurrency LLM Client with `asyncio.Semaphore` & Exponential Backoff
**Estimated Time:** 45 mins  
**Context:** Sending 200 concurrent requests to an external LLM API without rate limiting triggers HTTP 429 (`RateLimitError`) storms and thundering-herd retries.  
**Problem Statement:** Implement `BatchLLMClient.process_prompts(prompts: List[str]) -> List[str]`.
1. Enforce a maximum of `max_concurrency` in-flight requests at any moment using `asyncio.Semaphore`.
2. Call `await self.api.complete(prompt)`.
3. If `RateLimitError` is raised, retry up to `max_retries` times using exponential backoff: `sleep_time = base_delay * (2 ** attempt)` (where `attempt` starts at `0` for the first failure). Make sure the semaphore is released while sleeping during backoff so other requests can proceed.
4. Return results in the exact order of the input `prompts`. If a prompt exhausts all retries, raise the `RateLimitError`.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 5]
import asyncio
from typing import List

class RateLimitError(Exception):
    pass

class MockLLMProvider:
    def __init__(self, concurrency_limit: int = 3):
        self.concurrency_limit = concurrency_limit
        self.current_in_flight = 0
        self.max_observed_in_flight = 0
        self.call_counts = {}

    async def complete(self, prompt: str) -> str:
        self.current_in_flight += 1
        self.max_observed_in_flight = max(self.max_observed_in_flight, self.current_in_flight)
        self.call_counts[prompt] = self.call_counts.get(prompt, 0) + 1
        try:
            if self.current_in_flight > self.concurrency_limit:
                raise RateLimitError("Too many concurrent requests!")
            await asyncio.sleep(0.05)
            if "flaky" in prompt and self.call_counts[prompt] < 3:
                raise RateLimitError("Transient 429 Too Many Requests")
            return f"resp:{prompt}"
        finally:
            self.current_in_flight -= 1

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 5]
class BatchLLMClient:
    def __init__(self, api: MockLLMProvider, max_concurrency: int = 3, max_retries: int = 3, base_delay: float = 0.02):
        self.api = api
        self.semaphore = asyncio.Semaphore(max_concurrency)
        self.max_retries = max_retries
        self.base_delay = base_delay

    async def _fetch_single(self, prompt: str) -> str:
        # YOUR CODE HERE
        pass

    async def process_prompts(self, prompts: List[str]) -> List[str]:
        # YOUR CODE HERE
        pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 5]
async def validate_ex5():
    provider = MockLLMProvider(concurrency_limit=3)
    client = BatchLLMClient(api=provider, max_concurrency=3, max_retries=3, base_delay=0.02)
    prompts = [f"prompt_{i}" if i % 3 != 0 else f"flaky_{i}" for i in range(12)]
    
    results = await client.process_prompts(prompts)
    assert len(results) == 12
    assert results[0] == "resp:flaky_0"
    assert results[1] == "resp:prompt_1"
    assert provider.max_observed_in_flight <= 3, f"Semaphore failed! Max in-flight was {provider.max_observed_in_flight}"
    assert provider.call_counts["flaky_0"] == 3, "Flaky prompt should have succeeded on 3rd attempt"
    print("PASS: Exercise 5 Semaphore & Backoff pipeline verified.")

await validate_ex5()

<details><summary>Click to view solution (Exercise 5)</summary>

```python
class BatchLLMClient:
    def __init__(self, api: MockLLMProvider, max_concurrency: int = 3, max_retries: int = 3, base_delay: float = 0.02):
        self.api = api
        self.semaphore = asyncio.Semaphore(max_concurrency)
        self.max_retries = max_retries
        self.base_delay = base_delay

    async def _fetch_single(self, prompt: str) -> str:
        for attempt in range(self.max_retries + 1):
            try:
                async with self.semaphore:
                    return await self.api.complete(prompt)
            except RateLimitError:
                if attempt == self.max_retries:
                    raise
                await asyncio.sleep(self.base_delay * (2 ** attempt))

    async def process_prompts(self, prompts: List[str]) -> List[str]:
        return list(await asyncio.gather(*(self._fetch_single(p) for p in prompts)))
```
</details>

---
## Exercise 6: Bounded Producer-Consumer Pipeline with `asyncio.Queue`
**Estimated Time:** 45 mins  
**Context:** Unbounded task creation (`[asyncio.create_task(...) for item in million_items]`) exhausts RAM and crashes the process. Bounded `asyncio.Queue(maxsize=N)` provides natural backpressure: producers pause automatically when workers fall behind.  
**Problem Statement:** Implement `run_bounded_pipeline(items: List[int], num_workers: int, queue_size: int) -> Dict[int, int]`.
1. Create an `asyncio.Queue(maxsize=queue_size)`.
2. A producer coroutine pushes all `items` into the queue, followed by `None` sentinel values to shut down the workers cleanly.
3. Spawn exactly `num_workers` consumer tasks. Each consumer pulls an integer `x`, awaits `asyncio.sleep(0.02)`, computes `x * x`, and stores `results[x] = x * x`.
4. Return the completed `results` dictionary once all items are processed and all workers have exited cleanly.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 6]
import asyncio
import time
from typing import List, Dict, Optional

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 6]
async def run_bounded_pipeline(items: List[int], num_workers: int = 4, queue_size: int = 2) -> Dict[int, int]:
    """
    Processes items using a bounded asyncio.Queue with backpressure and clean worker shutdown.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 6]
async def validate_ex6():
    items = list(range(20))
    start = time.perf_counter()
    res = await run_bounded_pipeline(items, num_workers=4, queue_size=2)
    elapsed = time.perf_counter() - start
    
    assert len(res) == 20, f"Expected 20 results, got {len(res)}"
    for k in items:
        assert res[k] == k * k, f"Mismatch for {k}: got {res.get(k)}"
    assert elapsed < 0.25, f"Workers did not run concurrently! Elapsed: {elapsed:.3f}s"
    print(f"PASS: Exercise 6 bounded queue pipeline finished in {elapsed:.3f}s.")

await validate_ex6()

<details><summary>Click to view solution (Exercise 6)</summary>

```python
async def run_bounded_pipeline(items: List[int], num_workers: int = 4, queue_size: int = 2) -> Dict[int, int]:
    queue: asyncio.Queue[Optional[int]] = asyncio.Queue(maxsize=queue_size)
    results: Dict[int, int] = {}

    async def producer():
        for item in items:
            await queue.put(item)
        for _ in range(num_workers):
            await queue.put(None)

    async def worker():
        while True:
            item = await queue.get()
            try:
                if item is None:
                    break
                await asyncio.sleep(0.02)
                results[item] = item * item
            finally:
                queue.task_done()

    workers = [asyncio.create_task(worker()) for _ in range(num_workers)]
    await producer()
    await queue.join()
    await asyncio.gather(*workers)
    return results
```
</details>

---
## Exercise 7: FastAPI Lifespan State Management & Dependency Injection
**Estimated Time:** 45 mins  
**Context:** Creating a new `httpx.AsyncClient` or database pool on every incoming FastAPI request exhausts OS ephemeral ports (socket exhaustion) and adds TLS handshake latency. FastAPI uses an `@asynccontextmanager` `lifespan` function to initialize shared resources at startup and tear them down cleanly at shutdown.  
**Problem Statement:**
1. Write a `lifespan(app: FastAPI)` async context manager that initializes an instance of `MockAsyncPool()`, assigns it to global `last_created_pool` and `app.state.pool`, calls `await pool.connect()`, yields, and upon exit calls `await pool.close()`.
2. Write a dependency function `get_pool(request: Request) -> MockAsyncPool` that extracts `request.app.state.pool`.
3. Create a route `@app_ex7.get("/query")` that injects `pool: MockAsyncPool = Depends(get_pool)` and returns `{"db_status": await pool.ping()}`.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 7]
from contextlib import asynccontextmanager
from fastapi import FastAPI, Depends, Request
from typing import Optional
import httpx
import asyncio

class MockAsyncPool:
    def __init__(self):
        self.connected = False
        self.closed = False

    async def connect(self):
        await asyncio.sleep(0.01)
        self.connected = True

    async def ping(self) -> str:
        if not self.connected or self.closed:
            raise RuntimeError("Pool not active!")
        return "ALIVE"

    async def close(self):
        await asyncio.sleep(0.01)
        self.closed = True
        self.connected = False

last_created_pool: Optional[MockAsyncPool] = None

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 7]
@asynccontextmanager
async def lifespan(app: FastAPI):
    global last_created_pool
    # YOUR CODE HERE: instantiate MockAsyncPool, assign to last_created_pool & app.state.pool, connect, yield, and close
    pass

app_ex7 = FastAPI(lifespan=lifespan)

# YOUR CODE HERE: Define get_pool dependency and GET "/query" endpoint on app_ex7


In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 7]
async def validate_ex7():
    async with app_ex7.router.lifespan_context(app_ex7):
        assert last_created_pool is not None, "Pool was not created"
        assert last_created_pool.connected is True, "Pool was not connected on startup"
        
        transport = httpx.ASGITransport(app=app_ex7)
        async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
            resp = await client.get("/query")
            assert resp.status_code == 200
            assert resp.json() == {"db_status": "ALIVE"}
            
    assert last_created_pool.closed is True, "Pool was not closed on lifespan teardown!"
    print("PASS: Exercise 7 FastAPI lifespan startup and teardown verified.")

await validate_ex7()

<details><summary>Click to view solution (Exercise 7)</summary>

```python
@asynccontextmanager
async def lifespan(app: FastAPI):
    global last_created_pool
    pool = MockAsyncPool()
    last_created_pool = pool
    await pool.connect()
    app.state.pool = pool
    try:
        yield
    finally:
        await pool.close()

app_ex7 = FastAPI(lifespan=lifespan)

def get_pool(request: Request) -> MockAsyncPool:
    return request.app.state.pool

@app_ex7.get("/query")
async def query_endpoint(pool: MockAsyncPool = Depends(get_pool)):
    return {"db_status": await pool.ping()}
```
</details>

---
## Exercise 8: Streaming LLM Tokens via Server-Sent Events (SSE) & Disconnect Cleanup
**Estimated Time:** 45 mins  
**Context:** When streaming tokens from an LLM via FastAPI `StreamingResponse(..., media_type="text/event-stream")`, if the client closes their browser tab mid-generation, the backend must detect the disconnect/cancellation and immediately release upstream GPU/LLM connections.  
**Problem Statement:** Implement an async generator `sse_token_generator(tokens: List[str], tracker: StreamTracker)` and mount it on `@app_ex8.get("/stream")`.
- For each token in `tokens`, sleep `0.02s`, increment `tracker.tokens_emitted += 1`, and yield an SSE-formatted frame: `f"data: {json.dumps({'token': token})}\n\n"`.
- After all tokens, yield `"data: [DONE]\n\n"`.
- Ensure that inside a `try ... finally` block, `tracker.stream_closed = True` is always set even if the generator is closed early (`await gen.aclose()`).

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 8]
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from typing import List, AsyncGenerator
import json
import asyncio
import httpx

class StreamTracker:
    def __init__(self):
        self.tokens_emitted = 0
        self.stream_closed = False

app_ex8 = FastAPI()

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 8]
async def sse_token_generator(tokens: List[str], tracker: StreamTracker) -> AsyncGenerator[str, None]:
    # YOUR CODE HERE
    pass

@app_ex8.get("/stream")
async def stream_llm():
    tracker = StreamTracker()
    app_ex8.state.last_tracker = tracker
    tokens = ["Async", " ", "FastAPI", " ", "rocks!"]
    return StreamingResponse(sse_token_generator(tokens, tracker), media_type="text/event-stream")

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 8]
async def validate_ex8():
    transport = httpx.ASGITransport(app=app_ex8)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
        chunks = []
        async with client.stream("GET", "/stream") as response:
            assert response.status_code == 200
            assert "text/event-stream" in response.headers["content-type"]
            async for line in response.aiter_lines():
                if line:
                    chunks.append(line)
        assert chunks[0] == 'data: {"token": "Async"}'
        assert chunks[-1] == "data: [DONE]"
        assert app_ex8.state.last_tracker.stream_closed is True

    early_tracker = StreamTracker()
    gen = sse_token_generator(["t1", "t2", "t3", "t4"], early_tracker)
    first_chunk = await anext(gen)
    assert 't1' in first_chunk
    await gen.aclose()
    assert early_tracker.tokens_emitted == 1
    assert early_tracker.stream_closed is True, "Generator failed to clean up on early disconnect!"
    print("PASS: Exercise 8 SSE formatting and disconnect cleanup verified.")

await validate_ex8()

<details><summary>Click to view solution (Exercise 8)</summary>

```python
async def sse_token_generator(tokens: List[str], tracker: StreamTracker) -> AsyncGenerator[str, None]:
    try:
        for token in tokens:
            await asyncio.sleep(0.02)
            tracker.tokens_emitted += 1
            yield f"data: {json.dumps({'token': token})}\n\n"
        yield "data: [DONE]\n\n"
    finally:
        tracker.stream_closed = True

@app_ex8.get("/stream")
async def stream_llm():
    tracker = StreamTracker()
    app_ex8.state.last_tracker = tracker
    tokens = ["Async", " ", "FastAPI", " ", "rocks!"]
    return StreamingResponse(sse_token_generator(tokens, tracker), media_type="text/event-stream")
```
</details>

---
## Exercise 9: Bi-Directional WebSockets with Pydantic Validation & Dead-Socket Pruning
**Estimated Time:** 45 mins  
**Context:** Real-time voice/chat agent backends use WebSockets for full-duplex communication. A single broken client socket during a broadcast must not crash the broadcast loop for healthy connected clients.  
**Problem Statement:** Implement `ConnectionManager` and a WebSocket message handler:
1. `ConnectionManager.broadcast(message: dict)` sends JSON to all `active_connections`. If any `ws.send_json(message)` raises an exception (`WebSocketDisconnect` or `RuntimeError`), remove that broken WebSocket from `active_connections` and continue delivering to remaining clients.
2. Implement `handle_ws_message(raw_data: dict) -> dict` which validates `raw_data` using the strict Pydantic model `WSIncomingMessage(action: str, content: str)`. If valid, return `{"status": "ok", "echo": msg.content}`. If `ValidationError` is raised, return `{"status": "error", "detail": "Invalid schema"}` without closing the socket.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 9]
from fastapi import WebSocketDisconnect
from pydantic import BaseModel, ConfigDict, ValidationError
from typing import List, Any

class WSIncomingMessage(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")
    action: str
    content: str

class MockWebSocket:
    def __init__(self, should_fail: bool = False):
        self.should_fail = should_fail
        self.sent_messages = []
        self.accepted = False

    async def accept(self):
        self.accepted = True

    async def send_json(self, data: dict):
        if self.should_fail:
            raise WebSocketDisconnect(code=1006)
        self.sent_messages.append(data)

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 9]
class ConnectionManager:
    def __init__(self):
        self.active_connections: List[Any] = []

    async def connect(self, websocket: Any):
        await websocket.accept()
        self.active_connections.append(websocket)

    def disconnect(self, websocket: Any):
        if websocket in self.active_connections:
            self.active_connections.remove(websocket)

    async def broadcast(self, message: dict):
        # YOUR CODE HERE: send to all active_connections and prune any socket that raises an exception
        pass

def handle_ws_message(raw_data: dict) -> dict:
    # YOUR CODE HERE: validate with WSIncomingMessage and return appropriate dict
    pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 9]
async def validate_ex9():
    mgr = ConnectionManager()
    ws_good1 = MockWebSocket(should_fail=False)
    ws_dead = MockWebSocket(should_fail=True)
    ws_good2 = MockWebSocket(should_fail=False)

    await mgr.connect(ws_good1)
    await mgr.connect(ws_dead)
    await mgr.connect(ws_good2)
    assert len(mgr.active_connections) == 3

    await mgr.broadcast({"event": "token", "val": "hello"})
    assert len(mgr.active_connections) == 2, "Dead websocket was not pruned during broadcast!"
    assert ws_good1.sent_messages == [{"event": "token", "val": "hello"}]
    assert ws_good2.sent_messages == [{"event": "token", "val": "hello"}]

    valid_res = handle_ws_message({"action": "chat", "content": "hi"})
    assert valid_res == {"status": "ok", "echo": "hi"}

    invalid_res = handle_ws_message({"action": "chat", "content": 12345})
    assert invalid_res == {"status": "error", "detail": "Invalid schema"}
    print("PASS: Exercise 9 WebSocket ConnectionManager & schema validation verified.")

await validate_ex9()

<details><summary>Click to view solution (Exercise 9)</summary>

```python
class ConnectionManager:
    def __init__(self):
        self.active_connections: List[Any] = []

    async def connect(self, websocket: Any):
        await websocket.accept()
        self.active_connections.append(websocket)

    def disconnect(self, websocket: Any):
        if websocket in self.active_connections:
            self.active_connections.remove(websocket)

    async def broadcast(self, message: dict):
        dead_sockets = []
        for ws in list(self.active_connections):
            try:
                await ws.send_json(message)
            except Exception:
                dead_sockets.append(ws)
        for ws in dead_sockets:
            self.disconnect(ws)

def handle_ws_message(raw_data: dict) -> dict:
    try:
        msg = WSIncomingMessage.model_validate(raw_data)
        return {"status": "ok", "echo": msg.content}
    except ValidationError:
        return {"status": "error", "detail": "Invalid schema"}
```
</details>

---
## Exercise 10: Streaming NDJSON Validation with Pydantic v2 `TypeAdapter`
**Estimated Time:** 35 mins  
**Context:** Constructing a full wrapper `BaseModel` just to validate individual JSON lines in a high-throughput streaming response wastes CPU cycles. Pydantic v2's `TypeAdapter` compiles a standalone Rust validator for any type (`Union`, `TypedDict`, `Annotated`) once at module startup and validates raw JSON bytes directly via `.validate_json()`.  
**Problem Statement:** Create a module-level `TypeAdapter` for `StreamEvent` (`Union[ToolCallEvent, TextChunkEvent]` discriminated by `event_type`). Implement `consume_ndjson_stream(byte_stream: AsyncGenerator[bytes, None])` that parses each line using `EVENT_ADAPTER.validate_json(line)`. Return a tuple `(valid_events, error_count)`.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 10]
from pydantic import BaseModel, TypeAdapter, Field, ConfigDict, ValidationError
from typing import Literal, Union, Annotated, List, Tuple, AsyncGenerator
import asyncio

class ToolCallEvent(BaseModel):
    model_config = ConfigDict(strict=True)
    event_type: Literal["tool_call"]
    tool_name: str
    arguments: dict

class TextChunkEvent(BaseModel):
    model_config = ConfigDict(strict=True)
    event_type: Literal["text_chunk"]
    delta: str

StreamEvent = Annotated[Union[ToolCallEvent, TextChunkEvent], Field(discriminator="event_type")]

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 10]
EVENT_ADAPTER = None  # YOUR CODE HERE: Initialize TypeAdapter(StreamEvent)

async def consume_ndjson_stream(
    byte_stream: AsyncGenerator[bytes, None]
) -> Tuple[List[Union[ToolCallEvent, TextChunkEvent]], int]:
    """
    Validates each raw JSON byte line using EVENT_ADAPTER.validate_json().
    Counts lines that fail ValidationError without crashing the stream.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 10]
async def validate_ex10():
    async def mock_stream():
        lines = [
            b'{"event_type": "text_chunk", "delta": "Hello"}',
            b'{"event_type": "tool_call", "tool_name": "search", "arguments": {"q": "fastapi"}}',
            b'{"event_type": "text_chunk", "delta": 9999}',
            b'{"event_type": "unknown", "delta": "bad"}',
            b'{"event_type": "text_chunk", "delta": " World"}'
        ]
        for line in lines:
            await asyncio.sleep(0.005)
            yield line

    events, errors = await consume_ndjson_stream(mock_stream())
    assert len(events) == 3, f"Expected 3 valid events, got {len(events)}"
    assert errors == 2, f"Expected 2 validation errors, got {errors}"
    assert isinstance(events[0], TextChunkEvent)
    assert isinstance(events[1], ToolCallEvent)
    print("PASS: Exercise 10 Discriminated Union TypeAdapter stream validation passed.")

await validate_ex10()

<details><summary>Click to view solution (Exercise 10)</summary>

```python
EVENT_ADAPTER = TypeAdapter(StreamEvent)

async def consume_ndjson_stream(
    byte_stream: AsyncGenerator[bytes, None]
) -> Tuple[List[Union[ToolCallEvent, TextChunkEvent]], int]:
    valid_events: List[Union[ToolCallEvent, TextChunkEvent]] = []
    error_count = 0
    async for raw_line in byte_stream:
        if not raw_line.strip():
            continue
        try:
            event = EVENT_ADAPTER.validate_json(raw_line)
            valid_events.append(event)
        except ValidationError:
            error_count += 1
    return valid_events, error_count
```
</details>

---
# Level 3: Edge Cases, Performance & Failure Modes

## Exercise 11: Event Loop Lag Watchdog & Detecting Hidden Sync Bottlenecks
**Estimated Time:** 45 mins  
**Context:** In production FastAPI servers, a developer might accidentally call a synchronous CPU-heavy function or synchronous `requests.get()` inside an `async def` endpoint. Standard request timers only show high latency for *all* requests without telling you that the event loop itself froze.  
**Problem Statement:**
1. Implement an event loop lag monitor `measure_max_loop_lag(coro, interval: float = 0.01) -> Tuple[Any, float]` that runs a background task waking up every `interval` seconds. Lag on each tick is `max(0.0, actual_elapsed - interval)`. When `coro` completes, cancel the monitor and return `(result, max_lag_observed)`.
2. Implement `optimized_pipeline(payloads: List[str]) -> List[int]` to replace `buggy_pipeline` by offloading `cpu_heavy_hash(p)` to a worker thread concurrently so that `max_lag_observed` drops below `0.04` seconds.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 11]
import hashlib
import time
import asyncio
from typing import List, Tuple, Any

def cpu_heavy_hash(data: str) -> int:
    """Blocks the calling thread for ~80ms."""
    t0 = time.perf_counter()
    h = data.encode()
    while time.perf_counter() - t0 < 0.08:
        h = hashlib.sha256(h).digest()
    return int.from_bytes(h[:4], "big")

async def buggy_pipeline(payloads: List[str]) -> List[int]:
    results = []
    for p in payloads:
        await asyncio.sleep(0.01)
        results.append(cpu_heavy_hash(p))
    return results

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 11]
async def measure_max_loop_lag(coro, interval: float = 0.01) -> Tuple[Any, float]:
    """
    Runs coro while monitoring event loop scheduling delay every `interval` seconds.
    Returns (coro_result, max_lag_seconds).
    """
    # YOUR CODE HERE
    pass

async def optimized_pipeline(payloads: List[str]) -> List[int]:
    """
    Refactored version of buggy_pipeline that runs concurrently and keeps event loop lag < 0.04s.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 11]
async def validate_ex11():
    payloads = ["doc1", "doc2", "doc3"]
    
    res_buggy, lag_buggy = await measure_max_loop_lag(buggy_pipeline(payloads))
    assert lag_buggy >= 0.06, f"Lag monitor failed to detect blocking call! Measured lag: {lag_buggy:.4f}s"

    res_opt, lag_opt = await measure_max_loop_lag(optimized_pipeline(payloads))
    assert res_opt == res_buggy, "Output hashes do not match!"
    assert lag_opt < 0.04, f"Optimized pipeline still blocked the event loop! Lag: {lag_opt:.4f}s"
    print(f"PASS: Exercise 11 verified. Buggy lag: {lag_buggy*1000:.1f}ms -> Optimized lag: {lag_opt*1000:.1f}ms")

await validate_ex11()

<details><summary>Click to view solution (Exercise 11)</summary>

```python
async def measure_max_loop_lag(coro, interval: float = 0.01) -> Tuple[Any, float]:
    max_lag = 0.0
    running = True

    async def watchdog():
        nonlocal max_lag
        last = time.perf_counter()
        while running:
            await asyncio.sleep(interval)
            now = time.perf_counter()
            lag = max(0.0, (now - last) - interval)
            if lag > max_lag:
                max_lag = lag
            last = now

    monitor_task = asyncio.create_task(watchdog())
    try:
        result = await coro
        return result, max_lag
    finally:
        running = False
        monitor_task.cancel()
        try:
            await monitor_task
        except asyncio.CancelledError:
            pass

async def optimized_pipeline(payloads: List[str]) -> List[int]:
    async def process_one(p: str) -> int:
        await asyncio.sleep(0.01)
        return await asyncio.to_thread(cpu_heavy_hash, p)

    return list(await asyncio.gather(*(process_one(p) for p in payloads)))
```
</details>

---
## Exercise 12: Automated CPU Bottleneck Profiling with `cProfile` & `pstats`
**Estimated Time:** 45 mins  
**Context:** When an async service exhibits high CPU utilization, guessing which function is responsible wastes engineering hours. Using `cProfile.Profile()` programmatically with `pstats.Stats` lets you pinpoint the exact function with the highest internal (`tottime`) CPU cost and optimize it.  
**Problem Statement:**
1. Implement `find_top_cpu_culprit(coro) -> str` which profiles `await coro` using `cProfile.Profile()`, inspects `pstats.Stats(profiler).stats`, and returns the function name (`func_tuple[2]`) with the highest internal execution time (`tottime`, which is index `2` of the stat value tuple `(cc, nc, tt, ct, callers)`).
2. Implement `fast_token_counter(texts: List[str]) -> List[int]` to replace the $O(N^2)$ list lookup in `slow_token_counter` with an $O(N)$ `set` operation so execution finishes at least **5x faster**.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 12]
import cProfile
import pstats
import time
import asyncio
from typing import List

def _inefficient_unique_count(tokens: List[str]) -> int:
    seen = []
    for t in tokens:
        if t not in seen:
            seen.append(t)
    return len(seen)

async def slow_token_counter(texts: List[str]) -> List[int]:
    await asyncio.sleep(0.01)
    return [_inefficient_unique_count(t.split()) for t in texts]

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 12]
async def find_top_cpu_culprit(coro) -> str:
    """
    Profiles the given coroutine using cProfile and returns the name of the function
    with the highest internal execution time ('tottime').
    """
    # YOUR CODE HERE
    pass

async def fast_token_counter(texts: List[str]) -> List[int]:
    """
    Optimized O(N) replacement for slow_token_counter.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 12]
async def validate_ex12():
    sample_texts = [" ".join([f"word_{i}" for i in range(4000)]) for _ in range(3)]
    
    t0 = time.perf_counter()
    culprit = await find_top_cpu_culprit(slow_token_counter(sample_texts))
    slow_time = time.perf_counter() - t0
    assert culprit == "_inefficient_unique_count", f"Expected '_inefficient_unique_count', got '{culprit}'"

    t1 = time.perf_counter()
    fast_res = await fast_token_counter(sample_texts)
    fast_time = time.perf_counter() - t1

    assert fast_res == [4000, 4000, 4000]
    assert fast_time < slow_time / 5, f"Optimization insufficient: slow={slow_time:.3f}s, fast={fast_time:.3f}s"
    print(f"PASS: Identified culprit '{culprit}' and sped up execution from {slow_time:.3f}s to {fast_time:.4f}s.")

await validate_ex12()

<details><summary>Click to view solution (Exercise 12)</summary>

```python
async def find_top_cpu_culprit(coro) -> str:
    profiler = cProfile.Profile()
    profiler.enable()
    try:
        await coro
    finally:
        profiler.disable()

    stats = pstats.Stats(profiler)
    top_func = max(stats.stats.items(), key=lambda item: item[1][2])
    return top_func[0][2]

async def fast_token_counter(texts: List[str]) -> List[int]:
    await asyncio.sleep(0.01)
    return [len(set(t.split())) for t in texts]
```
</details>

---
## Exercise 13: Diagnosing & Fixing Async Memory Leaks (`tracemalloc` & Fire-and-Forget Tasks)
**Estimated Time:** 45 mins  
**Context:** Two classic memory leaks in async Python microservices are:
1. Storing unbounded request/task references or large payloads in a global list/cache without pruning.
2. Calling `asyncio.create_task()` without storing and cleaning up the task via `task.add_done_callback(background_tasks.discard)`.  
**Problem Statement:** `LeakyBackgroundDispatcher` leaks both `Task` objects and large byte buffers on every `.dispatch()` call. Refactor `CleanBackgroundDispatcher` so that:
- In-flight tasks are stored in `self._active_tasks: set` to prevent premature garbage collection while running, and automatically discarded via `task.add_done_callback(self._active_tasks.discard)` upon completion.
- Only the 64-byte SHA256 hex digest of `payload` is retained in `self.completed_digests` (bounded to the most recent `max_history=50` items using `collections.deque(maxlen=max_history)`), allowing the raw 100KB payload buffers to be garbage collected immediately.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 13]
import tracemalloc
import gc
from collections import deque
import hashlib
import asyncio

class LeakyBackgroundDispatcher:
    def __init__(self):
        self.all_tasks = []
        self.raw_payload_history = []

    async def _worker(self, payload: bytes):
        await asyncio.sleep(0.01)
        self.raw_payload_history.append(payload)

    def dispatch(self, payload: bytes):
        t = asyncio.create_task(self._worker(payload))
        self.all_tasks.append(t)

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 13]
class CleanBackgroundDispatcher:
    def __init__(self, max_history: int = 50):
        # YOUR CODE HERE: initialize self._active_tasks (set) and self.completed_digests (deque)
        pass

    async def _worker(self, payload: bytes):
        # YOUR CODE HERE: sleep 0.01s, hash payload with sha256().hexdigest(), append to completed_digests
        pass

    def dispatch(self, payload: bytes):
        # YOUR CODE HERE: create task, add to _active_tasks, register _active_tasks.discard callback
        pass

    async def wait_all(self):
        if self._active_tasks:
            await asyncio.gather(*list(self._active_tasks))

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 13]
async def validate_ex13():
    gc.collect()
    tracemalloc.start()
    
    snap_before = tracemalloc.take_snapshot()
    dispatcher = CleanBackgroundDispatcher(max_history=50)
    
    for i in range(100):
        dispatcher.dispatch(b"x" * 100_000 + str(i).encode())
        
    await dispatcher.wait_all()
    gc.collect()
    snap_after = tracemalloc.take_snapshot()
    tracemalloc.stop()

    assert len(dispatcher._active_tasks) == 0, f"Leaked {len(dispatcher._active_tasks)} task references!"
    assert len(dispatcher.completed_digests) == 50, f"Expected 50 bounded digests, got {len(dispatcher.completed_digests)}"
    
    stats = snap_after.compare_to(snap_before, 'lineno')
    total_allocated_kb = sum(stat.size_diff for stat in stats) / 1024
    assert total_allocated_kb < 500, f"Memory leak detected! Retained {total_allocated_kb:.1f} KB (expected < 500 KB)"
    print(f"PASS: Exercise 13 memory-safe task lifecycle verified. Net retained memory: {total_allocated_kb:.1f} KB.")

await validate_ex13()

<details><summary>Click to view solution (Exercise 13)</summary>

```python
class CleanBackgroundDispatcher:
    def __init__(self, max_history: int = 50):
        self._active_tasks: set[asyncio.Task] = set()
        self.completed_digests: deque[str] = deque(maxlen=max_history)

    async def _worker(self, payload: bytes):
        await asyncio.sleep(0.01)
        digest = hashlib.sha256(payload).hexdigest()
        self.completed_digests.append(digest)

    def dispatch(self, payload: bytes):
        task = asyncio.create_task(self._worker(payload))
        self._active_tasks.add(task)
        task.add_done_callback(self._active_tasks.discard)

    async def wait_all(self):
        if self._active_tasks:
            await asyncio.gather(*list(self._active_tasks))
```
</details>

---
## Exercise 14: Shielding Critical State Mutations from Cancellation (`asyncio.shield`)
**Estimated Time:** 45 mins  
**Context:** When a caller wraps an endpoint or task in `asyncio.wait_for(coro, timeout=0.06)`, timeout expiration injects `CancelledError` at the currently awaited line. If your coroutine already deducted user credits from an LLM billing ledger and is halfway through committing the audit log, a raw cancellation corrupts billing state.  
**Problem Statement:** Implement `process_inference_with_safe_commit(prompt: str, ledger: BillingLedger) -> str`.
1. First await `slow_llm_generation(prompt)` (`0.04s`). If cancelled during generation, allow cancellation to propagate normally without billing the user.
2. Once generation succeeds, the commit phase (`ledger.atomic_debit_and_log(prompt, cost=10)`) MUST NOT be interrupted even if the outer caller cancels the task mid-commit. Use `asyncio.shield()` and handle `asyncio.CancelledError` during the shielded commit so that `atomic_debit_and_log` always finishes before re-raising `CancelledError`.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 14]
import asyncio

class BillingLedger:
    def __init__(self):
        self.credits_deducted = 0
        self.audit_logs_written = 0

    async def atomic_debit_and_log(self, prompt: str, cost: int):
        self.credits_deducted += cost
        await asyncio.sleep(0.06)
        self.audit_logs_written += 1

async def slow_llm_generation(prompt: str) -> str:
    await asyncio.sleep(0.04)
    return f"Result for {prompt}"

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 14]
async def process_inference_with_safe_commit(prompt: str, ledger: BillingLedger) -> str:
    """
    Runs slow_llm_generation(prompt). Once complete, shields ledger.atomic_debit_and_log(prompt, 10)
    so that even if cancelled during commit, the commit finishes before CancelledError is re-raised.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 14]
async def validate_ex14():
    ledger1 = BillingLedger()
    try:
        await asyncio.wait_for(process_inference_with_safe_commit("p1", ledger1), timeout=0.02)
    except asyncio.TimeoutError:
        pass
    assert ledger1.credits_deducted == 0 and ledger1.audit_logs_written == 0

    ledger2 = BillingLedger()
    try:
        await asyncio.wait_for(process_inference_with_safe_commit("p2", ledger2), timeout=0.06)
        assert False, "Should have raised TimeoutError"
    except asyncio.TimeoutError:
        pass
    
    assert ledger2.credits_deducted == 10, "Credits were not deducted!"
    assert ledger2.audit_logs_written == 1, "Partial commit corruption! Audit log was interrupted by cancellation."
    print("PASS: Exercise 14 asyncio.shield atomic commit protection verified.")

await validate_ex14()

<details><summary>Click to view solution (Exercise 14)</summary>

```python
async def process_inference_with_safe_commit(prompt: str, ledger: BillingLedger) -> str:
    result = await slow_llm_generation(prompt)
    commit_task = asyncio.create_task(ledger.atomic_debit_and_log(prompt, cost=10))
    try:
        await asyncio.shield(commit_task)
    except asyncio.CancelledError:
        await commit_task
        raise
    return result
```
</details>

---
## Exercise 15: Dynamic Concurrency Limiter with Fast-Fail Queue Shedding
**Estimated Time:** 45 mins  
**Context:** Standard `asyncio.Semaphore(5)` allows 10,000 requests to wait in the semaphore queue forever, driving tail latency (P99) to minutes while clients have already timed out. Production load shedders combine a concurrency limit (`max_active`) with a strict queue waiter limit (`max_waiters`) and a queue acquisition timeout.  
**Problem Statement:** Implement an async context manager class `AdaptiveLoadShedder(max_active: int, max_waiters: int, acquire_timeout: float)`:
1. If the semaphore is currently locked and `self.current_waiters >= self.max_waiters`, immediately raise `LoadShedError("Queue full - fast fail")` without waiting.
2. Otherwise, increment `self.current_waiters` and wait up to `acquire_timeout` seconds to acquire `self.semaphore`. If `TimeoutError` occurs while waiting, raise `LoadShedError("Queue wait timeout")`.
3. Ensure both `current_active` and `current_waiters` counters are accurately maintained and decremented inside `finally` blocks so counters never drift under exceptions or cancellations.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 15]
import asyncio
from contextlib import asynccontextmanager

class LoadShedError(Exception):
    pass

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 15]
class AdaptiveLoadShedder:
    def __init__(self, max_active: int, max_waiters: int, acquire_timeout: float = 0.05):
        self.max_active = max_active
        self.max_waiters = max_waiters
        self.acquire_timeout = acquire_timeout
        self.semaphore = asyncio.Semaphore(max_active)
        self.current_waiters = 0
        self.current_active = 0

    @asynccontextmanager
    async def acquire(self):
        # YOUR CODE HERE
        pass

In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 15]
async def validate_ex15():
    shedder = AdaptiveLoadShedder(max_active=2, max_waiters=2, acquire_timeout=0.05)
    outcomes = []

    async def simulated_request(req_id: int, hold_time: float):
        try:
            async with shedder.acquire():
                await asyncio.sleep(hold_time)
                outcomes.append(("success", req_id))
        except LoadShedError as e:
            outcomes.append(("shed", str(e)))

    await asyncio.gather(*(simulated_request(i, 0.15) for i in range(6)))

    successes = [o for o in outcomes if o[0] == "success"]
    fast_fails = [o for o in outcomes if o[0] == "shed" and "Queue full" in o[1]]
    timeouts = [o for o in outcomes if o[0] == "shed" and "timeout" in o[1]]

    assert len(successes) == 2, f"Expected 2 successes, got {len(successes)}"
    assert len(fast_fails) == 2, f"Expected 2 immediate queue-full fast-fails, got {len(fast_fails)}"
    assert len(timeouts) == 2, f"Expected 2 queue wait timeouts, got {len(timeouts)}"
    assert shedder.current_waiters == 0 and shedder.current_active == 0, "Counter leak detected!"
    print("PASS: Exercise 15 AdaptiveLoadShedder fast-fail & timeout shedding verified.")

await validate_ex15()

<details><summary>Click to view solution (Exercise 15)</summary>

```python
class AdaptiveLoadShedder:
    def __init__(self, max_active: int, max_waiters: int, acquire_timeout: float = 0.05):
        self.max_active = max_active
        self.max_waiters = max_waiters
        self.acquire_timeout = acquire_timeout
        self.semaphore = asyncio.Semaphore(max_active)
        self.current_waiters = 0
        self.current_active = 0

    @asynccontextmanager
    async def acquire(self):
        if self.semaphore.locked() and self.current_waiters >= self.max_waiters:
            raise LoadShedError("Queue full - fast fail")

        self.current_waiters += 1
        try:
            await asyncio.wait_for(self.semaphore.acquire(), timeout=self.acquire_timeout)
        except asyncio.TimeoutError:
            raise LoadShedError("Queue wait timeout")
        finally:
            self.current_waiters -= 1

        self.current_active += 1
        try:
            yield
        finally:
            self.current_active -= 1
            self.semaphore.release()
```
</details>

---
# Level 4: Milestone Mini-Project

## Exercise 16: Production Multi-Provider LLM Gateway Microservice
**Estimated Time:** 90+ mins  
**Context:** You are building the core LLM Gateway for an AI platform. It must validate incoming prompts strictly, enforce concurrency limits with fast-fail load shedding, route requests across a primary LLM provider with automatic failover to a backup provider using `asyncio.wait_for`, stream SSE chunks back to the client, and track real-time latency and token metrics inside the FastAPI `lifespan` state.

### Architecture Requirements:
1. **Strict Pydantic v2 Models (`strict=True`, `extra="forbid"`)**:
   - `GatewayRequest`:
     - `prompt`: `str` (strip whitespace via `'before'` validator; must be at least 3 chars after stripping).
     - `max_tokens`: `int` (`1 <= max_tokens <= 4096`).
     - `timeout_ms`: `int` (`50 <= timeout_ms <= 5000`).
2. **Lifespan State (`GatewayState`)**:
   - Initialized in `gateway_lifespan(app)` and stored at `app.state.gateway`.
   - Contains an `asyncio.Semaphore(3)` for concurrency limiting, counters for `total_requests`, `fallback_triggered`, `active_streams`, and `closed = False` (set `closed = True` on shutdown).
3. **Endpoints**:
   - `POST /v1/chat/stream`:
     - If `gateway.semaphore.locked()`, immediately raise `HTTPException(status_code=429, detail="Gateway at max concurrency")`.
     - Otherwise, acquire the semaphore for the duration of the SSE stream.
     - Attempt to fetch the token list from `await primary_llm_provider(req.prompt)` wrapped in `asyncio.wait_for(..., timeout=req.timeout_ms / 1000.0)`.
     - If `asyncio.TimeoutError` or `RuntimeError` occurs on the primary provider, increment `gateway.fallback_triggered += 1` and call `await backup_llm_provider(req.prompt)`.
     - Stream the resulting tokens via `StreamingResponse(..., media_type="text/event-stream")` where each token is formatted as `f"data: {json.dumps({'token': t})}\n\n"`, ending with `"data: [DONE]\n\n"`.
   - `GET /v1/metrics`:
     - Returns `{"total_requests": ..., "fallback_triggered": ..., "active_streams": ...}`.

In [ ]:
# [ENVIRONMENT / SETUP CELL - EXERCISE 16]
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, ConfigDict, Field, field_validator
from contextlib import asynccontextmanager
from typing import List, Any
import httpx
import asyncio
import json

async def primary_llm_provider(prompt: str) -> List[str]:
    if "slow_primary" in prompt:
        await asyncio.sleep(0.3)
    if "crash_primary" in prompt:
        raise RuntimeError("Primary GPU node OOM")
    await asyncio.sleep(0.02)
    return ["Primary:", " ", prompt]

async def backup_llm_provider(prompt: str) -> List[str]:
    await asyncio.sleep(0.02)
    return ["Backup:", " ", prompt]

In [ ]:
# [STUDENT IMPLEMENTATION CELL - EXERCISE 16]
class GatewayRequest(BaseModel):
    # YOUR CODE HERE
    pass

class GatewayState:
    def __init__(self, max_concurrency: int = 3):
        self.semaphore = asyncio.Semaphore(max_concurrency)
        self.total_requests = 0
        self.fallback_triggered = 0
        self.active_streams = 0
        self.closed = False

@asynccontextmanager
async def gateway_lifespan(app: FastAPI):
    # YOUR CODE HERE
    pass

app_gateway = FastAPI(lifespan=gateway_lifespan)

# YOUR CODE HERE: Implement POST "/v1/chat/stream" and GET "/v1/metrics"


In [ ]:
# [AUTOMATED VALIDATION CELL - EXERCISE 16]
async def validate_ex16():
    async with app_gateway.router.lifespan_context(app_gateway):
        transport = httpx.ASGITransport(app=app_gateway)
        async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
            
            bad_resp = await client.post("/v1/chat/stream", json={"prompt": "  ab ", "max_tokens": 100, "timeout_ms": 200})
            assert bad_resp.status_code == 422, f"Expected 422 for short prompt, got {bad_resp.status_code}"

            lines = []
            async with client.stream("POST", "/v1/chat/stream", json={"prompt": "  hello world  ", "max_tokens": 100, "timeout_ms": 200}) as resp:
                assert resp.status_code == 200
                async for line in resp.aiter_lines():
                    if line:
                        lines.append(line)
            assert lines[0] == 'data: {"token": "Primary:"}'
            assert lines[2] == 'data: {"token": "hello world"}'
            assert lines[-1] == "data: [DONE]"

            fb_lines = []
            async with client.stream("POST", "/v1/chat/stream", json={"prompt": "slow_primary_test", "max_tokens": 100, "timeout_ms": 80}) as resp:
                assert resp.status_code == 200
                async for line in resp.aiter_lines():
                    if line:
                        fb_lines.append(line)
            assert fb_lines[0] == 'data: {"token": "Backup:"}'

            async def make_concurrent_call():
                return await client.post("/v1/chat/stream", json={"prompt": "concurrent_call", "max_tokens": 50, "timeout_ms": 500})

            responses = await asyncio.gather(*(make_concurrent_call() for _ in range(5)))
            status_codes = [r.status_code for r in responses]
            assert 429 in status_codes, f"Expected HTTP 429 when exceeding concurrency=3, got {status_codes}"

            metrics_resp = await client.get("/v1/metrics")
            metrics = metrics_resp.json()
            assert metrics["fallback_triggered"] >= 1
            assert metrics["active_streams"] == 0, "Active streams counter leaked!"
            assert metrics["total_requests"] >= 5

    assert app_gateway.state.gateway.closed is True, "Gateway state was not marked closed on shutdown!"
    print("PASS: Milestone Exercise 16 Production LLM Gateway passed all integration checks!")

await validate_ex16()

<details><summary>Click to view solution (Exercise 16)</summary>

```python
class GatewayRequest(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")

    prompt: str
    max_tokens: int = Field(ge=1, le=4096)
    timeout_ms: int = Field(ge=50, le=5000)

    @field_validator("prompt", mode="before")
    @classmethod
    def clean_prompt(cls, v: Any) -> str:
        if not isinstance(v, str):
            raise ValueError("prompt must be a string")
        stripped = v.strip()
        if len(stripped) < 3:
            raise ValueError("prompt must be at least 3 characters after stripping")
        return stripped

class GatewayState:
    def __init__(self, max_concurrency: int = 3):
        self.semaphore = asyncio.Semaphore(max_concurrency)
        self.total_requests = 0
        self.fallback_triggered = 0
        self.active_streams = 0
        self.closed = False

@asynccontextmanager
async def gateway_lifespan(app: FastAPI):
    state = GatewayState(max_concurrency=3)
    app.state.gateway = state
    try:
        yield
    finally:
        state.closed = True

app_gateway = FastAPI(lifespan=gateway_lifespan)

def get_gateway(request: Request) -> GatewayState:
    return request.app.state.gateway

@app_gateway.post("/v1/chat/stream")
async def chat_stream(req: GatewayRequest, gateway: GatewayState = Depends(get_gateway)):
    if gateway.semaphore.locked():
        raise HTTPException(status_code=429, detail="Gateway at max concurrency")

    await gateway.semaphore.acquire()
    gateway.total_requests += 1
    gateway.active_streams += 1

    try:
        try:
            tokens = await asyncio.wait_for(
                primary_llm_provider(req.prompt),
                timeout=req.timeout_ms / 1000.0
            )
        except (asyncio.TimeoutError, RuntimeError):
            gateway.fallback_triggered += 1
            tokens = await backup_llm_provider(req.prompt)
    except Exception:
        gateway.active_streams -= 1
        gateway.semaphore.release()
        raise

    async def event_stream():
        try:
            for t in tokens:
                await asyncio.sleep(0.01)
                yield f"data: {json.dumps({'token': t})}\n\n"
            yield "data: [DONE]\n\n"
        finally:
            gateway.active_streams -= 1
            gateway.semaphore.release()

    return StreamingResponse(event_stream(), media_type="text/event-stream")

@app_gateway.get("/v1/metrics")
async def get_metrics(gateway: GatewayState = Depends(get_gateway)):
    return {
        "total_requests": gateway.total_requests,
        "fallback_triggered": gateway.fallback_triggered,
        "active_streams": gateway.active_streams,
    }
```
</details>